# Word Embeddings

Let's train a word vector model on ECHOE. `Word2Vec` is a commonly used model:

In [1]:
from pathlib import Path
from gensim.models import Word2Vec
from pprint import pprint
from git import Repo

In [2]:
# HTTPS clone point:
remote = 'https://github.com/ECHOEProject/echoe.git'
# Desired target folder name:
local = Path.cwd().parent / 'corpora' / 'echoe'
# Only clone if the target folder doesn't already exist:
if not(local.exists()):
    repo = Repo.clone_from(remote, local)
# Else, just update the working copy from remote:
else:
    repo = Repo(local)
    assert isinstance(repo, Repo)
    repo.remotes.origin.pull()
assert not repo.bare

For this exercise let's use the preprocessed plaintext corpus rather than running all that preprocessing on the XML corpus again (we have template notebooks for both approaches). Because the model requires tokenized sentences as input, we'll create a list of sentences (demarcated by newlines in ECHOE) prior to tokenization:

In [3]:
corpus = []
echoe_plaintext = Path(local / 'plaintext')
for i in echoe_plaintext.glob('*txt'):
    document = open(i).read().splitlines()
    for sentence in document:
        if ': ' in sentence:
            sentence_bare = sentence.split(': ', 1)[1]
        corpus.append(sentence_bare)
sentences = [[token for token in document.split()] for document in corpus]

Time to train our model. Take note of the arguments; you may want to make a copy of this notebook and try tweaking some of these hyperparameters. One of the most effective hyperparameters with which to experiment is `window`: as Jurafsky & Martin note, it can drastically alter the type of associative relationship between target and context word.

In [4]:
model = Word2Vec(sentences=sentences, min_count=1, vector_size=300, workers=2, window=3, epochs=30)

The `most_similar` method then reveals what terms have been inferred to be closest in meaning:

In [5]:
# "Deofol" means "devil":
model.wv.most_similar('deofol')

[('deoful', 0.7778957486152649),
 ('deofel', 0.58835768699646),
 ('diofol', 0.5427219271659851),
 ('sacerd', 0.5100679993629456),
 ('þeodfeond', 0.5047441124916077),
 ('mann', 0.4888206720352173),
 ('mon', 0.48390039801597595),
 ('antecrist', 0.4698840379714966),
 ('gediglian', 0.4638711214065552),
 ('cyningc', 0.46376025676727295)]

In [6]:
# "Niht" means "night":
model.wv.most_similar('niht')

[('wuca', 0.5568523406982422),
 ('writingfeðer', 0.5461229085922241),
 ('wucan', 0.5228034853935242),
 ('boc', 0.5150917172431946),
 ('underntid', 0.506930410861969),
 ('nihte', 0.4849965274333954),
 ('siþe', 0.4800070822238922),
 ('eastran', 0.47880542278289795),
 ('stowe', 0.4767691493034363),
 ('minstre', 0.4756581783294678)]

In [7]:
# "Mæden" means "Young woman, virgin":
model.wv.most_similar('mæden')

[('gebære', 0.6793105006217957),
 ('seow', 0.6765958070755005),
 ('mæidan', 0.6509431600570679),
 ('sancta', 0.6058911085128784),
 ('hælfe', 0.5672722458839417),
 ('beripede', 0.5664668679237366),
 ('egidius', 0.5631386041641235),
 ('fasten', 0.5617127418518066),
 ('þrigæ', 0.54282546043396),
 ('gehylde', 0.5400829911231995)]

In [8]:
# "Wop" means "weeping":
model.wv.most_similar('wop')

[('granung', 0.8430888652801514),
 ('wanung', 0.8081660866737366),
 ('toða', 0.8074253797531128),
 ('gristbitung', 0.8037125468254089),
 ('hungor', 0.7816916704177856),
 ('cwanung', 0.7766725420951843),
 ('toþa', 0.7746875882148743),
 ('gefeall', 0.7630771398544312),
 ('heof', 0.7607223987579346),
 ('singal', 0.7591626644134521)]

In [9]:
# "Blis" means "bliss":
model.wv.most_similar('blis')

[('myrhð', 0.769582211971283),
 ('bliss', 0.7536584138870239),
 ('gefea', 0.749565601348877),
 ('rest', 0.742129385471344),
 ('sargung', 0.6815195679664612),
 ('eadignes', 0.6760310530662537),
 ('med', 0.6603533625602722),
 ('mærð', 0.6573022603988647),
 ('unrotnes', 0.643825888633728),
 ('ar', 0.6340749859809875)]

We can also run arithmetic operations on our terms. Keep in mind that all these "distances" are cosines of the angles between two vectors, as obtained by way of their dot product:

In [10]:
man_child = str(round(model.wv.distance('wer', 'cild'), 2))
woman_child = str(round(model.wv.distance('wif', 'cild'), 2))
print("The distance between 'man' and 'child' is " + man_child + ", while the distance between 'woman' and 'child' is " + woman_child + ".")

The distance between 'man' and 'child' is 0.67, while the distance between 'woman' and 'child' is 0.42.


In [11]:
wop_gefea = str(round(model.wv.distance('wop', 'gefea'), 2))
gefea_blis = str(round(model.wv.distance('gefea', 'blis'), 2))
print("The distance between 'weeping' and 'joy' is " + wop_gefea + ", while the distance between 'joy' and 'bliss' is " + gefea_blis + ".")

The distance between 'weeping' and 'joy' is 0.49, while the distance between 'joy' and 'bliss' is 0.25.


So although that looks useful at first glance, unfortunately our corpus is too small to come to the understanding that different inflected forms of the same word should count as very similar:

In [12]:
model.wv.distance('þridde', 'þriddan')

0.8338154256343842

Although these are both forms of the word for "three", the distance is greater than that between "man" and "child".

What about that famous vector math?

In [13]:
# king - man + woman:
model.wv.similar_by_vector(model.wv['cyning'] - model.wv['wer'] + model.wv['wif'])

[('cyning', 0.7124764323234558),
 ('wif', 0.49436426162719727),
 ('scyppend', 0.42326515913009644),
 ('kyning', 0.4187282919883728),
 ('hergienne', 0.40616312623023987),
 ('awestenne', 0.39185649156570435),
 ('god', 0.3707999289035797),
 ('land', 0.36952269077301025),
 ('hælo', 0.3689591884613037),
 ('feorh', 0.3685891330242157)]

Well, those are all still words of glory, so that's something. And let's face it, queens were almost unheard of in early England. Let's try a few more:

In [14]:
# Satan - sin + blessed:
model.wv.similar_by_vector(model.wv['satanas'] - model.wv['synn'] + model.wv['eadig'])

[('eadig', 0.8070144057273865),
 ('onlyhted', 0.5452660322189331),
 ('gefullod', 0.5323697924613953),
 ('syndriges', 0.5315786004066467),
 ('ægðres', 0.5244362950325012),
 ('gefulhtnede', 0.5229310393333435),
 ('unrihtlic', 0.5205869674682617),
 ('toboren', 0.5101322531700134),
 ('calamus', 0.5076049566268921),
 ('godhold', 0.504858136177063)]

In [15]:
# Christ - holy/saint + sin:
model.wv.similar_by_vector(model.wv['crist'] - model.wv['halga'] + model.wv['synn'])

[('crist', 0.7133699059486389),
 ('hæle', 0.32118484377861023),
 ('æhta', 0.319135457277298),
 ('betwuhs', 0.31774598360061646),
 ('dreoge', 0.30603501200675964),
 ('hælo', 0.3008946478366852),
 ('þrowode', 0.2895219027996063),
 ('begyten', 0.2863323986530304),
 ('edhwyrft', 0.28628817200660706),
 ('helpe', 0.2862473726272583)]

 We're really just getting an intersection between the two fields in the former query, and no effect at all in the latter. We'll need to expand our corpus and/or use smarter methods!

Let's add the _Dictionary of Old English_ Corpus (DOEC) to our training data and see what that does for us (you'll have to copy in DOEC from the Oxford Text Archive to `..corpora/doec/` yourself, and then run `../templates/load_doec.ipynb` from an appropriate folder to generate a bare plaintext corpus, to reproduce this part of the notebook):

In [17]:
larger_corpus = []
doec_plaintext = Path.cwd().parent / 'corpora' / 'doec-bare'
combined_list = list(echoe_plaintext.glob('*txt')) + list(doec_plaintext.glob('*txt'))
for i in combined_list:
    document = open(i).read().splitlines()
    for sentence in document:
        larger_corpus.append(sentence)
all_sentences = [[token for token in document.split()] for document in larger_corpus]
larger_model = Word2Vec(sentences=all_sentences, min_count=1, vector_size=300, workers=2, window=3, epochs=30)

In [ ]:
comparisons = [
    ('oþþe', 'oððe'),
    ('þridde', 'þriddan'),
    ('winter', 'sumer'),
    ('wer', 'cild'),
    ('wif', 'cild'),
    ('halga', 'sanctus')
]

In [ ]:
for pair in comparisons:
    distance = larger_model.wv.distance(pair[0], pair[1])
    print(f"The distance between {pair[0]} and {pair[1]} is {distance}.")

The distance between oþþe and oððe is 0.3863319158554077.
The distance between þridde and þriddan is 0.7533413469791412.
The distance between winter and sumer is 0.5718085169792175.
The distance between wer and cild is 0.8415999114513397.
The distance between wif and cild is 0.4931555390357971.
The distance between halga and sanctus is 0.6036190688610077.


Disappointing! Still, let's run through those nearest matches again:

In [ ]:
concepts = ['deofol', 'niht', 'mæden', 'wop', 'blis']

In [ ]:
for concept in concepts:
    related = larger_model.wv.most_similar(concept)
    print(f"The terms most similar to {concept} are as follows:")
    pprint(related)

The terms most similar to deofol are as follows:
[('deoful', 0.6725849509239197),
 ('deofel', 0.5462384223937988),
 ('diofol', 0.4729384779930115),
 ('þeodfeond', 0.42817679047584534),
 ('dema', 0.4093954265117645),
 ('dioful', 0.40791085362434387),
 ('ceorl', 0.40508872270584106),
 ('antecrist', 0.4000932276248932),
 ('feond', 0.3983190059661865),
 ('þeof', 0.39607009291648865)]
The terms most similar to niht are as follows:
[('nihte', 0.4989641606807709),
 ('wucan', 0.4938156008720398),
 ('nyht', 0.4624868929386139),
 ('neaht', 0.4278337359428406),
 ('nihta', 0.4216686487197876),
 ('tida', 0.411024808883667),
 ('wintre', 0.39768725633621216),
 ('monað', 0.39451467990875244),
 ('scytran', 0.3900070786476135),
 ('winter', 0.3897213935852051)]
The terms most similar to mæden are as follows:
[('fæmne', 0.5610029101371765),
 ('cild', 0.5329961180686951),
 ('maria', 0.5014251470565796),
 ('nytwyrþe', 0.4527575969696045),
 ('egidius', 0.45221683382987976),
 ('bryd', 0.45000940561294556),
 (

Finally, let's do our vector math once more:

In [ ]:
concepts = [
    ('cyning', 'wer', 'wif'),
    ('satanas', 'synn', 'eadig'),
    ('crist', 'halga', 'synn')
]

for concept in concepts:
    print(f"{concept[0]} - {concept[1]} + {concept[2]}:")
    print(larger_model.wv.similar_by_vector(larger_model.wv[concept[0]] - larger_model.wv[concept[1]] + larger_model.wv[concept[2]]))

cyning - wer + wif:
[('wif', 0.5935571193695068), ('cyning', 0.5716564655303955), ('cing', 0.381704181432724), ('cining', 0.3759607672691345), ('cyng', 0.34773963689804077), ('magas', 0.3371494710445404), ('kyning', 0.3309439718723297), ('cild', 0.32527559995651245), ('land', 0.3202517330646515), ('broþur', 0.3188495635986328)]
satanas - synn + eadig:
[('eadig', 0.8496550917625427), ('eadi', 0.5188611149787903), ('rihtwisosta', 0.4474625289440155), ('oferspræcea', 0.44427964091300964), ('arwierþa', 0.4396943151950836), ('foreðancula', 0.4381844103336334), ('fæsthydigne', 0.4303337633609772), ('wonhidig', 0.424898236989975), ('ofersprecola', 0.42454710602760315), ('beciped', 0.41433167457580566)]
crist - halga + synn:
[('crist', 0.6605784296989441), ('synn', 0.3542681932449341), ('geþeode', 0.2957265377044678), ('onbyrdnes', 0.27562475204467773), ('hellefyr', 0.2738836705684662), ('geþanc', 0.27365249395370483), ('uðwitan', 0.2671515643596649), ('beswicen', 0.26498955488204956), ('hwa',

So maybe our takeaway should be that with this model at this modest corpus size, we can identify the "nearest" concepts with some success, but not identify what inflectional forms belong together as the same word, while vector math is a tricky proposition in any language, but certainly a small one.